In [3]:
import torch  
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time
import math

In [4]:

VOCAB_SIZE = 64 
D_MODEL = 48
N_HEADS = 4 
N_LAYERS = 3 
D_FF = 96
MAX_LEN = 2048 

PROMPT = [1, 2, 3, 4, 5]
CORRECTNESS_CHECK_TOKENS = 30
GENERATION_LENGHTS = [10, 20, 40, 80, 160, 320]

REAL_LAYERS = 32
REAL_HEADS = 32
REAL_HEAD_DIM = 128 
REAL_BATCH = 32 
REAL_CONTEXT = 4096



In [20]:
torch.manual_seed(42)

class MaskedMultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // self.n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)


    def forward(self, x, past_kv = None, use_cache = False):
        B, T, C = x.shape 
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        if past_kv is not None:
            past_k, past_v = past_kv
            k = torch.cat([past_k, k], dim = 2) ## append the k in this step into the cache 
            v = torch.cat([past_v, v], dim = 2) ## append the v in this step into the cache 

        present_kv = (k, v) if use_cache else None

        attention = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        ## the Tk, and the Tq are what make the KV cache effective
        ## Tk = Cached token + New token 
        ## Tq = New token
        Tq, Tk = attention.shape[-2], attention.shape[-1]
        mask = torch.triu(torch.ones(Tq, Tk, dtype = torch.bool, device = x.device), diagonal = Tk - Tq + 1)
        ## this does, Tk x Tq and return False/True. True = not allowed to see and False = allowed to see
        attention = attention.masked_fill(mask, float("-inf"))
        attention = F.softmax(attention, dim = -1)

        out = attention @ v 
        out = out.transpose(1, 2).contiguous().view(B, Tq, C)
        return self.out_proj(out), present_kv


class PositionWiseFNN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(0.05)
        
    def forward(self, x):
        return self.dropout(self.fc2(F.relu(self.fc1(x))))

class Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.att = MaskedMultiHeadSelfAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.fnn = PositionWiseFNN(d_model, d_ff)



    def forward(self, x, past_kv = None, use_cache = False):
        attention_out, present_kv = self.att(self.ln1(x), past_kv, use_cache)
        x = x + attention_out
        x = x + self.fnn(self.ln2(x))
        return x, present_kv


In [21]:
class tinyGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_len):
        super().__init__()
        self.tok_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias = False)

    def forward(self, idx, past_key_values = None, use_cache = False):
        B, T = idx.shape
        offset = past_key_values[0][0].shape[2] if past_key_values is not None else 0
        pos = torch.arange(offset, offset + T, device = idx.device)
        x = self.tok_embed(idx) + self.pos_embed(pos)[None, :, :]

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None 
            x, present = block(x, past, use_cache)
            if use_cache:
                presents.append(present)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits, presents


model = tinyGPT(VOCAB_SIZE, D_MODEL, N_HEADS, N_LAYERS, D_FF, MAX_LEN).eval()

print(f"TinyGPT: {sum(p.numel() for p in model.parameters()):,} parameters")


TinyGPT: 161,424 parameters


### Generation with no Cache

In [22]:
@torch.no_grad()
def generate_no_cache(model, prompt_ids, n_new_tokens):
    tokens = list(prompt_ids)
    for _ in range(n_new_tokens):
        idx = torch.tensor([tokens], dtype = torch.long) ## convert it from list int into tensor 

        logits, = model(idx, use_cache = False)
        next_id = int(torch.argmax(logits[0, -1]))
        tokens.append(next_id)

        return tokens 

    

### Generation with KV Cache

In [23]:
@torch.no_grad()

def generation_with_cache(model, prompt_ids, n_new_tokens):
    tokens = list[prompt_ids]

    idx = torch.tensor([tokens], dtype = torch.long)
    logits, past = model(idx, use_cache = True)
    next_id = int(torch.argmax(logits[0, -1]))
    tokens.append(next_id)

    for _ in range(n_new_tokens):
        idx = torch.tensor([[tokens[-1]]], drype = torch.long)
        logits, past = model(idx, past_key_values = past, use_cache = True)
        next_id = int(torch.argmax(logits[0, -1]))
        tokens.append(next_id)

        return tokens